In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import MultiHeadAttention, LayerNormalization, Dropout, Layer
from tensorflow.keras.layers import Embedding, Input, GlobalAveragePooling1D, Dense
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.datasets import imdb
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.optimizers import Adam
import numpy as np
import pandas as pd
import warnings
import os
import pickle
warnings.filterwarnings("ignore", category=np.VisibleDeprecationWarning)
from itertools import chain
import string
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, auc, confusion_matrix,roc_auc_score
# import matplotlib.pyplot as plt
# import plotly.figure_factory as ff

In [2]:
# !pip uninstall -y numpy
# !pip install numpy==1.19.2 -- user


In [3]:
import nltk
#nltk.download("punkt")
nltk.download('stopwords')
from nltk.lm.preprocessing import pad_sequence
from nltk.tokenize import word_tokenize 
from nltk.corpus import stopwords


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\tommy\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [4]:
import wandb

In [5]:
file = "bitext_customer_support"

In [6]:
labels_column = "intent"
text_column = "response"

In [7]:
wandb.init(project=file, sync_tensorboard=True, save_code = True)
wandb.config["more"] = "custom"

wandb: Currently logged in as: tommytsuma7. Use `wandb login --relogin` to force relogin


In [8]:
from wandb.integration.keras import WandbCallback, WandbMetricsLogger , WandbModelCheckpoint 

In [9]:
df = pd.read_csv(f"D://CoreOutline/data/{file}.csv")

In [10]:
df

,Unnamed: 0,flags,instruction,category,intent,response
0,0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...
3,3,BL,I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with can...
4,4,BCELN,"I cannot afford this order, cancel purchase {{...",ORDER,cancel_order,I'm sensitive to the fact that you're facing f...
...,...,...,...,...,...,...
26867,26867,BL,I am waiting for a rebate of {{Refund Amount}}...,REFUND,track_refund,Thank you for sharing your situation regarding...
26868,26868,BIL,how to see if there is anything wrong with my ...,REFUND,track_refund,Ensuring the accuracy of your restitution is o...
26869,26869,BLQZ,I'm waiting for a reimbjrsement of {{Currency ...,REFUND,track_refund,"Firstly, I genuinely understand the importance..."
26870,26870,BL,I don't know what to do to see my reimbursemen...,REFUND,track_refund,I've understood you're unsure about how to che...


In [11]:
class TransformerBlock(Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super(TransformerBlock, self).__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.ff_dim = ff_dim
        self.att = MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = Sequential(
            [Dense(ff_dim, activation="relu"), 
             Dense(embed_dim),]
        )
        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)
        self.dropout1 = Dropout(rate)
        self.dropout2 = Dropout(rate)

    def call(self, inputs):
        attn_output = self.att(inputs, inputs)
        attn_output = self.dropout1(attn_output)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output)
        return self.layernorm2(out1 + ffn_output)
    def get_config(self):
        config = super().get_config().copy()
        config.update({
            'embed_dim': self.embed_dim,
            'num_heads': self.num_heads,
            'ff_dim': self.ff_dim
        })
        return config

In [12]:
class TokenAndPositionEmbedding(Layer):
    def __init__(self, maxlen, vocab_size, embed_dim):
        super(TokenAndPositionEmbedding, self).__init__()
        self.token_emb = Embedding(input_dim=vocab_size, output_dim=embed_dim)
        self.pos_emb = Embedding(input_dim=maxlen, output_dim=embed_dim)
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim

    def call(self, x):
        maxlen = tf.shape(x)[-1]
        positions = tf.range(start=0, limit=maxlen, delta=1)
        positions = self.pos_emb(positions)
        x = self.token_emb(x)
        return x + positions
    def get_config(self):
        config = super().get_config().copy()
        config.update({
            'vocab_size': self.vocab_size,
            'embed_dim': self.embed_dim,
        })
        return config

In [13]:
vocab_size = 20000  # Only consider the top 20k words
maxlen = 518  # Only consider the first 200 words of each movie review

(x_train, y_train), (x_val, y_val) = imdb.load_data(num_words=vocab_size)
print(len(x_train), "Training sequences")
print(len(x_val), "Validation sequences")

25000 Training sequences
25000 Validation sequences


In [14]:
x_train = tf.keras.preprocessing.sequence.pad_sequences(x_train, maxlen=maxlen)
x_val = tf.keras.preprocessing.sequence.pad_sequences(x_val, maxlen=maxlen)

In [15]:
embed_dim = 32  # Embedding size for each token
num_heads = 2  # Number of attention heads
ff_dim = 32  # Hidden layer size in feed forward network inside transformer

inputs = Input(shape=(maxlen,))
embedding_layer = TokenAndPositionEmbedding(maxlen, vocab_size, embed_dim)
x = embedding_layer(inputs)
transformer_block = TransformerBlock(embed_dim, num_heads, ff_dim)
x = transformer_block(x)
x = GlobalAveragePooling1D()(x)
x = Dropout(0.1)(x)
x = Dense(20, activation="relu")(x)
x = Dropout(0.1)(x)
outputs = Dense(len(df[labels_column].unique()), activation="softmax")(x)



In [16]:
def remove_punctuation(x):
    return "".join([i for i in x if i not in string.punctuation])

In [17]:
df['no_punctuation_conversation'] = df[text_column].apply(remove_punctuation)

In [18]:
df['lowercase_conversation'] = df.no_punctuation_conversation.str.lower()

In [19]:
df['tokenized_conversation'] = [word_tokenize(i) for i in df['lowercase_conversation']]

In [20]:
def remove_stopwords(tokens):
    stopwords = nltk.corpus.stopwords.words('english')
    return [token for token in tokens if token not in stopwords]

In [21]:
df['no_stopwords_conversation'] = df['tokenized_conversation'].apply(remove_stopwords)

In [22]:
df['no_stopwords_conversation']

0        [ive, understood, question, regarding, canceli...
1        [ive, informed, question, canceling, order, or...
2        [sense, youre, seeking, assistance, canceling,...
3        [understood, need, assistance, canceling, purc...
4        [im, sensitive, fact, youre, facing, financial...
                               ...                        
26867    [thank, sharing, situation, regarding, pending...
26868    [ensuring, accuracy, restitution, utmost, impo...
26869    [firstly, genuinely, understand, importance, e...
26870    [ive, understood, youre, unsure, check, status...
26871    [completely, understandable, want, stay, updat...
Name: no_stopwords_conversation, Length: 26872, dtype: object

In [23]:
words = np.array(df['no_stopwords_conversation'])

In [24]:
words = list(chain(*words))

In [25]:
le = LabelEncoder()
df['issue_area_le'] = le.fit_transform(df[labels_column])

In [26]:
def word_map_tokenizer(text_arr):
    tokenizer = Tokenizer(num_words = 512, oov_token = '<00V>')
    tokenizer.fit_on_texts(text_arr)
    return tokenizer

In [27]:
tokenizer = word_map_tokenizer(words)

In [28]:
pad_token = 0  # The token to use for padding
max_length = max(len(seq) for seq in df['no_stopwords_conversation']) 

In [29]:
max_length

221

In [30]:
df['conversation_encoded'] = [list(chain(*tokenizer.texts_to_sequences(i))) for i in df['no_stopwords_conversation'] ]

In [31]:
df['conversation_encoded'] 

0        [206, 1, 1, 111, 242, 6, 6, 10, 7, 4, 11, 17, ...
1        [206, 373, 1, 242, 6, 6, 10, 7, 5, 3, 280, 1, ...
2        [1, 22, 217, 18, 242, 33, 33, 10, 6, 10, 135, ...
3        [1, 17, 18, 242, 33, 6, 10, 6, 10, 135, 157, 2...
4        [7, 1, 498, 22, 152, 1, 71, 17, 162, 33, 6, 10...
                               ...                        
26867    [41, 1, 180, 111, 1, 264, 1, 1, 376, 24, 160, ...
26868    [127, 414, 1, 182, 196, 8, 56, 1, 1, 1, 104, 1...
26869    [1, 1, 24, 196, 268, 153, 1, 1, 376, 1, 1, 376...
26870    [206, 1, 22, 284, 112, 104, 153, 52, 308, 7, 5...
26871    [202, 1, 175, 481, 398, 1, 1, 441, 1, 1, 495, ...
Name: conversation_encoded, Length: 26872, dtype: object

In [32]:
[len(i) for i in df['conversation_encoded'] if len(i) > 500]

[]

In [33]:
df['conversation_padded'] = [list(pad_sequence(i, n=(519-len(i)), pad_left=True, left_pad_symbol=pad_token)) for i in df['conversation_encoded']]

In [34]:
df['conversation_padded']

0        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
1        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
2        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
3        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
4        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
                               ...                        
26867    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
26868    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
26869    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
26870    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
26871    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
Name: conversation_padded, Length: 26872, dtype: object

In [35]:
[len(i) for i in df['conversation_padded']]

[518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518

In [36]:
X = np.array(pd.DataFrame([ i for i in df['conversation_padded']]))
y=  np.array(df['issue_area_le'])

In [37]:
X_train, X_test_val, y_train, y_test_val = train_test_split(X,y)

In [38]:
X_test, X_val, y_test, y_val = train_test_split(X_test_val,y_test_val)

In [39]:
len(X_test)

5038

In [40]:
np.shape(y_train)

(20154,)

In [41]:
np.shape(X_train)

(20154, 518)

In [42]:
callbacks = []

In [43]:
callbacks.append(WandbMetricsLogger())

In [44]:
earlystopping = EarlyStopping(monitor="val_accuracy",
                             patience=10,
                             min_delta=0,
                             mode='min',
                             restore_best_weights=False,
                             baseline=None,
                             verbose=0)
# callbacks.append(earlystopping)

In [45]:
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_accuracy',
                                                 factor=0.5,
                                                 patience=30,
                                                 min_lr=0.000001,
                                                 cooldown=5)

callbacks.append(reduce_lr)

In [46]:
checkpoint_path = "training_2/cp-{epoch:04d}.keras"
checkpoint_dir = os.path.dirname(checkpoint_path)

cp_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_path, 
    verbose=1, 
    save_weights_only=False
    )

callbacks.append(cp_callback)

In [47]:
model = Model(inputs=inputs, outputs=outputs)


In [48]:
model.compile(optimizer=Adam(learning_rate=0.01), loss="sparse_categorical_crossentropy", metrics=["accuracy"])


In [ ]:
history = model.fit(X_train, y_train, 
                    batch_size=32, epochs=200, 
                    validation_data=(X_val, y_val),
                    callbacks=callbacks
                   )

Epoch 1/200
630/630 [==============================] - ETA: 14:19 - loss: 3.4796 - accuracy: 0.0000e+0 - ETA: 3:28 - loss: 3.4178 - accuracy: 0.0156    - ETA: 3:34 - loss: 3.4123 - accuracy: 0.01 - ETA: 3:32 - loss: 3.4012 - accuracy: 0.01 - ETA: 3:27 - loss: 3.3937 - accuracy: 0.01 - ETA: 3:28 - loss: 3.3870 - accuracy: 0.01 - ETA: 3:27 - loss: 3.3810 - accuracy: 0.01 - ETA: 3:24 - loss: 3.3756 - accuracy: 0.02 - ETA: 3:25 - loss: 3.3716 - accuracy: 0.02 - ETA: 3:26 - loss: 3.3679 - accuracy: 0.02 - ETA: 3:25 - loss: 3.3650 - accuracy: 0.02 - ETA: 3:24 - loss: 3.3627 - accuracy: 0.02 - ETA: 3:24 - loss: 3.3604 - accuracy: 0.02 - ETA: 3:25 - loss: 3.3579 - accuracy: 0.03 - ETA: 3:25 - loss: 3.3557 - accuracy: 0.03 - ETA: 3:25 - loss: 3.3538 - accuracy: 0.03 - ETA: 3:23 - loss: 3.3523 - accuracy: 0.03 - ETA: 3:22 - loss: 3.3509 - accuracy: 0.03 - ETA: 3:20 - loss: 3.3496 - accuracy: 0.03 - ETA: 3:19 - loss: 3.3484 - accuracy: 0.03 - ETA: 3:19 - loss: 3.3473 - accuracy: 0.03 - ETA: 3:18 

In [ ]:
# for accuracy, loss in zip(history.history['val_accuracy'], history.history['val_loss']):
#     wandb.log({"accuracy": accuracy, "loss": loss})
# wandb.finish()

In [ ]:
y_preds = model.predict(X_test)

In [ ]:
y_preds = [ np.where(i==max(i)) for i in y_preds]

In [ ]:
y_preds = [i[0] for i in list(chain(*y_preds))]

In [ ]:
model.save_weights(f"D://CoreOutline/transformer/models/{file}.weights.h5")
results = model.evaluate(X_val, y_val, verbose=2)

for name, value in zip(model.metrics_names, results):
    print("%s: %.3f" % (name, value))

In [ ]:
print(classification_report(y_test, y_preds))

In [ ]:
cf_matrix = confusion_matrix(y_test, y_preds)

In [ ]:
y_preds_ = le.inverse_transform(y_preds)
y_test_ = le.inverse_transform(y_test)

In [ ]:
def plotPredictionConfusionMatrix(df, columns):
    fig = ff.create_annotated_heatmap(z=np.array(df),x=[i for i in columns], y=[i for i in columns], colorscale='blues', showscale=False, reversescale=False)
    fig['layout']['xaxis'].update(side='bottom', title='Actual')
    fig['layout']['yaxis'].update(side='left', title='Predicted')
    fig.update_layout(title=f'Prediction Confusion Matrix Heatmap', width=1000, height=1000)
    fig.write_image(f"D://CoreOutline/transformer/visualizations/{file}_heatmap.png")

In [ ]:
plotPredictionConfusionMatrix(cf_matrix, np.unique(y_preds_))

In [ ]:
custom_objects = {"TokenAndPositionEmbedding": TokenAndPositionEmbedding, "TransformerBlock":TransformerBlock}

In [ ]:
#pickle.dump(custom_objects, open("D://CoreOutline/transformer/models/custom_object.dict","wb"))

In [ ]:
model.save_model("D://CoreOutline/transformer/models")

In [ ]:
model

In [ ]:
pickle.dump(history, open(f"D://CoreOutline/transformer/history/{file}_history","wb"))

In [ ]:
pickle.dump(tokenizer, open(f"D://CoreOutline/transformer/tokens/{file}_tokenizer","wb"))

In [ ]:
# plt.plot(history.history['val_accuracy'])
# plt.ylabel('Validation Accuracy')
# plt.xlabel('Epochs')
# plt.savefig(f"D://CoreOutline/transformer/visualizations/{file}_accuracy_line_plot.png")
# plt.clf()

In [ ]:
# plt.plot(history.history['val_loss'])
# plt.ylabel('Validation Loss')
# plt.xlabel('Epochs')
# plt.savefig(f"D://CoreOutline/transformer/visualizations/{file}_loss_line_plot.png")
# plt.clf()